In [ ]:
!pip install httpx pandas numpy matplotlib seaborn scikit-learn

# Agentic Trader — Exploratory Data Analysis
# NARRATIVE: brief intro — purpose of this notebook, assets covered, data source (Binance public API)

## Section 1: Data Collection

In [ ]:
import httpx
import pandas as pd
import numpy as np

KLINES_URL = "https://api.binance.com/api/v3/klines"
_KLINES_COLS = [
    "open_time", "open", "high", "low", "close", "volume",
    "close_time", "quote_asset_volume", "num_trades",
    "taker_buy_base", "taker_buy_quote", "ignore",
]


def fetch_daily(symbol: str, limit: int = 365) -> pd.DataFrame:
    """Fetch daily OHLCV candles from Binance for a given symbol."""
    resp = httpx.get(
        KLINES_URL,
        params={"symbol": symbol, "interval": "1d", "limit": limit},
        timeout=15.0,
    )
    resp.raise_for_status()
    df = pd.DataFrame(resp.json(), columns=_KLINES_COLS)
    df["timestamp"] = pd.to_datetime(df["open_time"], unit="ms")
    for col in ("open", "high", "low", "close", "volume"):
        df[col] = df[col].astype(float)
    return df[["timestamp", "open", "high", "low", "close", "volume"]].set_index("timestamp")


btc = fetch_daily("BTCUSDT")
eth = fetch_daily("ETHUSDT")
bnb = fetch_daily("BNBUSDT")

print("BTC shape:", btc.shape)
btc.head()

In [ ]:
print("ETH shape:", eth.shape)
print("BNB shape:", bnb.shape)

# NARRATIVE: note on data — 365 daily candles per asset, Binance public endpoint, no API key required

## Section 2: Cleaning

In [ ]:
# Null value check
print("=== Null counts before ffill ===")
for name, df in [("BTC", btc), ("ETH", eth), ("BNB", bnb)]:
    print(f"\n{name}:")
    print(df.isnull().sum())

In [ ]:
# OHLCV integrity: high >= low
for name, df in [("BTC", btc), ("ETH", eth), ("BNB", bnb)]:
    violations = df[df["high"] < df["low"]]
    if violations.empty:
        print(f"{name}: no high < low violations")
    else:
        print(f"{name}: {len(violations)} violation(s)!")
        print(violations)

In [ ]:
# Apply forward-fill and confirm null counts after
btc = btc.ffill()
eth = eth.ffill()
bnb = bnb.ffill()

print("=== Null counts after ffill ===")
for name, df in [("BTC", btc), ("ETH", eth), ("BNB", bnb)]:
    print(f"{name}: {df.isnull().sum().sum()} nulls remaining")

# NARRATIVE: data quality — Binance daily data is generally clean; ffill is a precaution for any gaps from exchange downtime

## Section 3: Descriptive Statistics

In [ ]:
def describe_returns(df: pd.DataFrame, name: str) -> dict:
    returns = df["close"].pct_change().dropna()
    return {
        "mean_return": returns.mean(),
        "std": returns.std(),
        "skewness": returns.skew(),
        "kurtosis": returns.kurtosis(),
    }

stats = pd.DataFrame(
    {
        "BTC": describe_returns(btc, "BTC"),
        "ETH": describe_returns(eth, "ETH"),
        "BNB": describe_returns(bnb, "BNB"),
    }
).round(6)

stats

# NARRATIVE: interpret stats — compare mean returns, higher kurtosis indicates fat tails / extreme moves more likely than normal distribution

## Section 4: Time-Series Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.rcParams.update({
    "figure.facecolor": "#0d0d0d",
    "axes.facecolor": "#1a1a1a",
    "axes.edgecolor": "#2a2a2a",
    "axes.labelcolor": "#cccccc",
    "xtick.color": "#888888",
    "ytick.color": "#888888",
    "grid.color": "#2a2a2a",
    "text.color": "#cccccc",
    "lines.color": "#00ff88",
})

fig, (ax_price, ax_vol) = plt.subplots(2, 1, figsize=(14, 7), sharex=True,
                                        gridspec_kw={"height_ratios": [3, 1]})
fig.suptitle("BTC/USDT — Daily Price & Volume (365 days)", color="#cccccc", fontsize=14)

ax_price.plot(btc.index, btc["close"], color="#00ff88", linewidth=1.2, label="Close")
ax_price.fill_between(btc.index, btc["low"], btc["high"], alpha=0.15, color="#00ff88")
ax_price.set_ylabel("Price (USDT)", fontsize=10)
ax_price.grid(True, alpha=0.3)
ax_price.legend(loc="upper left", framealpha=0.2)

ax_vol.bar(btc.index, btc["volume"], color="#4488ff", alpha=0.6, width=0.8, label="Volume")
ax_vol.set_ylabel("Volume", fontsize=10)
ax_vol.set_xlabel("Date", fontsize=10)
ax_vol.grid(True, alpha=0.3)
ax_vol.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax_vol.xaxis.set_major_locator(mdates.MonthLocator(interval=2))

fig.autofmt_xdate(rotation=30)
plt.tight_layout()
plt.show()

# NARRATIVE: chart observations — identify notable price regimes, volume spikes correlating with price moves

## Section 5: Volatility Analysis

In [ ]:
# Rolling 30-day std of daily returns
btc_returns = btc["close"].pct_change()
rolling_vol = btc_returns.rolling(30).std()

# ATR(14) — manual computation
prev_close = btc["close"].shift(1)
tr = pd.concat(
    [
        btc["high"] - btc["low"],
        (btc["high"] - prev_close).abs(),
        (btc["low"] - prev_close).abs(),
    ],
    axis=1,
).max(axis=1)
atr14 = tr.ewm(com=13, adjust=False).mean()

fig, ax1 = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor("#0d0d0d")
ax1.set_facecolor("#1a1a1a")

color_vol = "#ffaa00"
ax1.set_xlabel("Date", color="#cccccc")
ax1.set_ylabel("30-day Rolling Volatility (σ)", color=color_vol)
ax1.plot(btc.index, rolling_vol, color=color_vol, linewidth=1.2, label="30d Volatility")
ax1.tick_params(axis="y", labelcolor=color_vol)
ax1.tick_params(colors="#888888")
ax1.grid(True, alpha=0.3, color="#2a2a2a")

ax2 = ax1.twinx()
ax2.set_facecolor("#1a1a1a")
color_atr = "#ff4444"
ax2.set_ylabel("ATR(14) (USDT)", color=color_atr)
ax2.plot(btc.index, atr14, color=color_atr, linewidth=1.2, alpha=0.8, label="ATR(14)")
ax2.tick_params(axis="y", labelcolor=color_atr)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", framealpha=0.2)

fig.suptitle("BTC Volatility — 30-day Rolling Std vs ATR(14)", color="#cccccc", fontsize=13)
fig.autofmt_xdate(rotation=30)
plt.tight_layout()
plt.show()

# NARRATIVE: volatility regimes — periods of elevated ATR signal higher risk; the rule engine uses ATR/price > 3% as HIGH volatility threshold

## Section 6: Correlation Matrix

In [ ]:
import seaborn as sns

returns_df = pd.DataFrame({
    "BTC": btc["close"].pct_change(),
    "ETH": eth["close"].pct_change(),
    "BNB": bnb["close"].pct_change(),
}).dropna()

corr = returns_df.corr(method="pearson")

fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor("#0d0d0d")
ax.set_facecolor("#1a1a1a")

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    vmin=-1, vmax=1,
    linewidths=0.5,
    linecolor="#2a2a2a",
    annot_kws={"size": 14, "color": "white"},
    ax=ax,
)
ax.set_title("Pearson Correlation — Daily Returns (BTC / ETH / BNB)",
             color="#cccccc", fontsize=12, pad=12)
ax.tick_params(colors="#cccccc")
plt.tight_layout()
plt.show()

print("\nCorrelation matrix:")
print(corr.round(3))

# NARRATIVE: correlation findings — high positive correlation between BTC and ETH/BNB typical of crypto market; implications for portfolio diversification

## Section 7: K-Means Clustering

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Fetch BTC 1h candles (500 candles)
resp_1h = httpx.get(
    KLINES_URL,
    params={"symbol": "BTCUSDT", "interval": "1h", "limit": 500},
    timeout=15.0,
)
resp_1h.raise_for_status()
btc1h = pd.DataFrame(resp_1h.json(), columns=_KLINES_COLS)
btc1h["timestamp"] = pd.to_datetime(btc1h["open_time"], unit="ms")
for col in ("open", "high", "low", "close", "volume"):
    btc1h[col] = btc1h[col].astype(float)
btc1h = btc1h.set_index("timestamp")
print("BTC 1h shape:", btc1h.shape)

In [ ]:
# --- Feature computation (manual pandas, no pandas-ta) ---

close_1h = btc1h["close"]

# RSI(14)
delta = close_1h.diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.ewm(com=13, adjust=False).mean()
avg_loss = loss.ewm(com=13, adjust=False).mean()
rs = avg_gain / avg_loss.replace(0, float("nan"))
rsi_1h = 100 - (100 / (1 + rs))

# MACD histogram (12, 26, 9)
ema12 = close_1h.ewm(span=12, adjust=False).mean()
ema26 = close_1h.ewm(span=26, adjust=False).mean()
macd_line_1h = ema12 - ema26
signal_line_1h = macd_line_1h.ewm(span=9, adjust=False).mean()
macd_hist_1h = macd_line_1h - signal_line_1h

# Volume vs 20-period average
volume_avg = btc1h["volume"].rolling(20).mean()
volume_vs_avg = btc1h["volume"] / volume_avg

# Assemble feature DataFrame and drop NaNs
features_df = pd.DataFrame({
    "rsi": rsi_1h,
    "macd_hist": macd_hist_1h,
    "volume_vs_avg": volume_vs_avg,
}).dropna()

print("Feature matrix shape:", features_df.shape)
features_df.describe().round(4)

In [ ]:
# Standardise and cluster
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_df)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
features_df = features_df.copy()
features_df["cluster"] = kmeans.fit_predict(X_scaled)

print("Cluster sizes:")
print(features_df["cluster"].value_counts().sort_index())
print("\nMean feature values per cluster:")
print(features_df.groupby("cluster")[["rsi", "macd_hist", "volume_vs_avg"]].mean().round(4))

In [ ]:
CLUSTER_COLORS = ["#00ff88", "#ff4444", "#ffaa00"]

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("#0d0d0d")
ax.set_facecolor("#1a1a1a")

for cluster_id, color in enumerate(CLUSTER_COLORS):
    mask = features_df["cluster"] == cluster_id
    ax.scatter(
        features_df.loc[mask, "rsi"],
        features_df.loc[mask, "macd_hist"],
        c=color, alpha=0.6, s=20,
        label=f"Cluster {cluster_id} (n={mask.sum()})",
    )

ax.axhline(0, color="#444444", linewidth=0.8, linestyle="--")
ax.axvline(50, color="#444444", linewidth=0.8, linestyle="--")
ax.set_xlabel("RSI (14)", color="#cccccc")
ax.set_ylabel("MACD Histogram", color="#cccccc")
ax.set_title("K-Means Clusters (k=3) — BTC 1h Candles\nFeatures: RSI, MACD Histogram, Volume vs Avg",
             color="#cccccc", fontsize=12)
ax.tick_params(colors="#888888")
ax.grid(True, alpha=0.2, color="#2a2a2a")
ax.legend(framealpha=0.2, labelcolor="#cccccc")

plt.tight_layout()
plt.show()

# NARRATIVE: cluster interpretation — characterise each cluster as e.g. oversold/high-volume, overbought/low-momentum, neutral; relate to rule engine signal types

## Section 8: Feature Engineering

In [ ]:
# Use BTC daily data for feature engineering examples
close = btc["close"].copy()
high = btc["high"].copy()
low = btc["low"].copy()

# ── RSI ─────────────────────────────────────────────────────────────────────
# Step-by-step manual implementation
delta = close.diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
# Wilder's smoothing = EWM with com = period - 1
avg_gain_manual = gain.ewm(com=13, adjust=False).mean()
avg_loss_manual = loss.ewm(com=13, adjust=False).mean()
rs_manual = avg_gain_manual / avg_loss_manual.replace(0, float("nan"))
rsi_manual = 100 - (100 / (1 + rs_manual))

# Pandas one-liner version (same formula)
def rsi_oneliner(s: pd.Series, period: int = 14) -> pd.Series:
    d = s.diff()
    return 100 - (100 / (1 + d.clip(lower=0).ewm(com=period-1, adjust=False).mean()
                             / (-d.clip(upper=0)).ewm(com=period-1, adjust=False).mean()))

rsi_quick = rsi_oneliner(close)

assert np.allclose(rsi_manual.dropna(), rsi_quick.dropna(), equal_nan=True), \
    "RSI: manual vs one-liner mismatch!"
print(f"RSI(14) — last value: {rsi_manual.iloc[-1]:.4f}  ✓ matches one-liner")

In [ ]:
# ── MACD ─────────────────────────────────────────────────────────────────────
# Step-by-step
ema12_manual = close.ewm(span=12, adjust=False).mean()
ema26_manual = close.ewm(span=26, adjust=False).mean()
macd_manual  = ema12_manual - ema26_manual
signal_manual = macd_manual.ewm(span=9, adjust=False).mean()
hist_manual  = macd_manual - signal_manual

# One-liner
macd_q = close.ewm(span=12, adjust=False).mean() - close.ewm(span=26, adjust=False).mean()
hist_q = macd_q - macd_q.ewm(span=9, adjust=False).mean()

assert np.allclose(hist_manual.dropna(), hist_q.dropna(), equal_nan=True), \
    "MACD histogram: manual vs one-liner mismatch!"
print(f"MACD Histogram — last value: {hist_manual.iloc[-1]:.6f}  ✓ matches one-liner")

In [ ]:
# ── Bollinger Bands ──────────────────────────────────────────────────────────
period_bb = 20
std_dev = 2.0

# Step-by-step
sma_manual = close.rolling(period_bb).mean()
std_manual = close.rolling(period_bb).std()
upper_manual = sma_manual + std_dev * std_manual
lower_manual = sma_manual - std_dev * std_manual
bandwidth_manual = (upper_manual - lower_manual) / sma_manual

# One-liner
sma_q = close.rolling(period_bb).mean()
std_q = close.rolling(period_bb).std()
upper_q = sma_q + std_dev * std_q
lower_q = sma_q - std_dev * std_q

assert np.allclose(upper_manual.dropna(), upper_q.dropna(), equal_nan=True), \
    "BB upper: manual vs one-liner mismatch!"
assert np.allclose(lower_manual.dropna(), lower_q.dropna(), equal_nan=True), \
    "BB lower: manual vs one-liner mismatch!"

current_price = close.iloc[-1]
bb_pos = (
    "above" if current_price > upper_manual.iloc[-1]
    else "below" if current_price < lower_manual.iloc[-1]
    else "inside"
)
print(f"BB(20,2) — upper={upper_manual.iloc[-1]:.2f}, lower={lower_manual.iloc[-1]:.2f}")
print(f"Current price {current_price:.2f} is {bb_pos} the bands  ✓")

# NARRATIVE: feature engineering rationale — RSI captures momentum exhaustion, MACD identifies trend direction changes, Bollinger Bands measure price relative to recent volatility; all three feed the rule engine

## Section 9: Backtesting

In [ ]:
# Fetch BTC 1h candles (1000 candles) for backtesting
resp_bt = httpx.get(
    KLINES_URL,
    params={"symbol": "BTCUSDT", "interval": "1h", "limit": 1000},
    timeout=15.0,
)
resp_bt.raise_for_status()
bt = pd.DataFrame(resp_bt.json(), columns=_KLINES_COLS)
bt["timestamp"] = pd.to_datetime(bt["open_time"], unit="ms")
for col in ("open", "high", "low", "close", "volume"):
    bt[col] = bt[col].astype(float)
bt = bt.set_index("timestamp").ffill()
print("Backtest data shape:", bt.shape)

In [ ]:
# ── Compute indicators inline (no agentic_trader import) ─────────────────────
c = bt["close"]

# RSI(14)
_d = c.diff()
_g = _d.clip(lower=0).ewm(com=13, adjust=False).mean()
_l = (-_d.clip(upper=0)).ewm(com=13, adjust=False).mean()
bt_rsi = 100 - (100 / (1 + _g / _l.replace(0, float("nan"))))

# MACD histogram (12, 26, 9)
_m = c.ewm(span=12, adjust=False).mean() - c.ewm(span=26, adjust=False).mean()
bt_macd_hist = _m - _m.ewm(span=9, adjust=False).mean()

# EMA(20)
bt_ema20 = c.ewm(span=20, adjust=False).mean()

# MACD histogram crossover flags
hist_prev = bt_macd_hist.shift(1)
crosses_positive = (bt_macd_hist > 0) & (hist_prev <= 0)  # was ≤ 0, now > 0
crosses_negative = (bt_macd_hist < 0) & (hist_prev >= 0)  # was ≥ 0, now < 0

# Signal classification — rule engine thresholds
sig_buy  = (bt_rsi < 35)  & crosses_positive & (c > bt_ema20)
sig_sell = (bt_rsi > 65)  & crosses_negative & (c < bt_ema20)

buy_indices  = bt.index[sig_buy]
sell_indices = bt.index[sig_sell]

print(f"STRONG_BUY signals:  {len(buy_indices)}")
print(f"STRONG_SELL signals: {len(sell_indices)}")
print(f"Total signals:       {len(buy_indices) + len(sell_indices)}")

In [ ]:
# ── Forward-4 evaluation ─────────────────────────────────────────────────────
FORWARD = 4
results = []

for idx in buy_indices:
    loc = bt.index.get_loc(idx)
    if loc + FORWARD >= len(bt):
        continue
    entry = bt["close"].iloc[loc]
    exit_  = bt["close"].iloc[loc + FORWARD]
    hit    = exit_ > entry
    pnl    = exit_ - entry
    results.append({"signal": "STRONG_BUY", "entry": entry, "exit": exit_, "hit": hit, "pnl": pnl})

for idx in sell_indices:
    loc = bt.index.get_loc(idx)
    if loc + FORWARD >= len(bt):
        continue
    entry = bt["close"].iloc[loc]
    exit_  = bt["close"].iloc[loc + FORWARD]
    hit    = exit_ < entry
    pnl    = entry - exit_
    results.append({"signal": "STRONG_SELL", "entry": entry, "exit": exit_, "hit": hit, "pnl": pnl})

results_df = pd.DataFrame(results)
print(results_df.head(10).to_string(index=False))

In [ ]:
# ── Hit rate summary ─────────────────────────────────────────────────────────
if len(results_df) == 0:
    print("No signals found — try a different time window or check VPN/data fetch.")
else:
    total     = len(results_df)
    n_correct = results_df["hit"].sum()
    overall   = 100 * n_correct / total

    buy_df  = results_df[results_df["signal"] == "STRONG_BUY"]
    sell_df = results_df[results_df["signal"] == "STRONG_SELL"]

    def hit_str(df: pd.DataFrame) -> str:
        if len(df) == 0:
            return "N/A (0 signals)"
        n = df["hit"].sum()
        return f"{100*n/len(df):.1f}% ({n}/{len(df)})"

    print(f"Signal hit rate: {overall:.1f}% ({int(n_correct)}/{total} signals correct)")
    print(f"  STRONG_BUY:  {hit_str(buy_df)}")
    print(f"  STRONG_SELL: {hit_str(sell_df)}")
    total_pnl = results_df["pnl"].sum()
    print(f"\nSimulated P&L: {total_pnl:+,.2f} USDT (illustrative, 1-unit positions, no fees)")

In [ ]:
# ── Equity curve ─────────────────────────────────────────────────────────────
if len(results_df) > 0:
    equity = results_df["pnl"].cumsum().reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(12, 5))
    fig.patch.set_facecolor("#0d0d0d")
    ax.set_facecolor("#1a1a1a")

    line_color = "#00ff88" if equity.iloc[-1] >= 0 else "#ff4444"
    ax.plot(equity.index, equity.values, color=line_color, linewidth=1.5, label="Equity curve")
    ax.fill_between(equity.index, 0, equity.values,
                    where=equity.values >= 0, alpha=0.15, color="#00ff88")
    ax.fill_between(equity.index, 0, equity.values,
                    where=equity.values < 0,  alpha=0.15, color="#ff4444")
    ax.axhline(0, color="#888888", linewidth=1.0, linestyle="--")

    ax.set_xlabel("Signal #", color="#cccccc")
    ax.set_ylabel("Cumulative P&L (USDT)", color="#cccccc")
    ax.set_title(
        f"Simulated Equity Curve — BTC 1h Rule Engine Signals\n"
        f"{total} signals, {overall:.1f}% hit rate, "
        f"final P&L: {total_pnl:+,.0f} USDT (1-unit, no fees)",
        color="#cccccc", fontsize=11,
    )
    ax.tick_params(colors="#888888")
    ax.grid(True, alpha=0.2, color="#2a2a2a")
    ax.legend(framealpha=0.2, labelcolor="#cccccc")

    plt.tight_layout()
    plt.show()

# NARRATIVE: limitations — illustrative only, no slippage/fees, not optimised for profitability, small signal count